# Trabalho Prático III: Recomendador de Disciplinas com NLP e Busca Semântica
**Disciplina:** Fundamentos de Inteligência Artificial (FIA) - Engenharia de Sistemas (UFMG)

Este notebook é **100% autossuficiente**, ou seja, ele baixa todos os dados necessários direto do repositório oficial no GitHub e carrega todas as funções na própria memória, sem necessidade de importar scripts externos.

In [ ]:
# 1. Download dos dados e Instalação de Bibliotecas Necessárias
!pip install -q sentence-transformers pandas numpy torch datasets accelerate
!mkdir -p data
!wget -qO data/disciplinas_engsis.json https://raw.githubusercontent.com/AndreRezende1999/tp3-ufmg-recomendador/main/data/disciplinas_engsis.json
!wget -qO data/dificuldade_engsis.csv https://raw.githubusercontent.com/AndreRezende1999/tp3-ufmg-recomendador/main/data/dificuldade_engsis.csv

print("Dados baixados com sucesso e salvos na pasta 'data/'!")

### 2. Definição das Funções do Recomendador
Abaixo estão todas as funções para:
1. Carregar a Base Curada.
2. Extrair o contexto semântico.
3. Utilizar o **Sentence-BERT** para realizar busca.
4. Aplicar regras de **Pré-Requisitos** e **Limite de Dificuldade**.

In [ ]:
import pandas as pd
import numpy as np
import torch
import json
import warnings
from sentence_transformers import SentenceTransformer, util

warnings.filterwarnings('ignore')

# ----------------- CONFIGURAÇÕES -----------------
DISCIPLINAS_JSON = "data/disciplinas_engsis.json"
MODEL_NAME = "neuralmind/bert-base-portuguese-cased"

# ----------------- CURADORIA DE DADOS -----------------
def get_discipline_table() -> pd.DataFrame:
    with open(DISCIPLINAS_JSON, 'r', encoding='utf-8') as f:
        return pd.DataFrame(json.load(f))

def get_discipline_text(row: pd.Series) -> str:
    nome = row.get("nome", "")
    area = row.get("area_conhecimento", "")
    ementa = row.get("ementa", "")
    return f"{nome}. Área: {area}. {ementa}"

def get_completed_profile_text(df: pd.DataFrame, completed_codes: list) -> str:
    text_parts = []
    for code in completed_codes:
        disc = df[df["codigo"] == code]
        if not disc.empty:
            text_parts.append(f"{disc.iloc[0]['nome']} ({disc.iloc[0]['area_conhecimento']})")
    return "Disciplinas cursadas com interesse: " + ", ".join(text_parts) + "."

# ----------------- SBERT ENCODER -----------------
def load_sbert_model() -> SentenceTransformer:
    return SentenceTransformer(MODEL_NAME)

def encode_texts(model, texts: list, batch_size=32) -> np.ndarray:
    return model.encode(texts, batch_size=batch_size, show_progress_bar=True, convert_to_numpy=True)

def encode_disciplines(model, df: pd.DataFrame) -> np.ndarray:
    texts = df.apply(get_discipline_text, axis=1).tolist()
    return encode_texts(model, texts)

def semantic_search(query_embedding: np.ndarray, corpus_embeddings: np.ndarray, top_k=10):
    query_tensor = torch.tensor(query_embedding)
    corpus_tensor = torch.tensor(corpus_embeddings)
    hits = util.semantic_search(query_tensor, corpus_tensor, top_k=top_k)[0]
    for hit in hits:
        hit['index'] = hit.pop('corpus_id')
    return hits

# ----------------- REGRAS DE RECOMENDAÇÃO -----------------
def prerequisites_satisfied(candidate_row, completed_codes) -> bool:
    prerequisites = set(candidate_row.get("pre_requisitos", []) or [])
    return prerequisites.issubset(set(completed_codes))

def filter_recommended_candidates(candidates: pd.DataFrame, completed_codes) -> pd.DataFrame:
    if candidates.empty: return candidates.copy()
    mask = candidates.apply(lambda row: prerequisites_satisfied(row, completed_codes), axis=1)
    return candidates.loc[mask].copy()

def greedy_balance_by_difficulty(candidates: pd.DataFrame, difficulty_budget: float, max_courses=None) -> pd.DataFrame:
    if candidates.empty: return candidates.copy()
    ordered = candidates.sort_values(["predicted_probability", "difficulty"], ascending=[False, True]).copy()
    
    selected_rows = []
    used_difficulty = 0.0
    for _, row in ordered.iterrows():
        difficulty = row.get("difficulty")
        numeric_diff = float(difficulty) if pd.notna(difficulty) else 0.0
        if max_courses is not None and len(selected_rows) >= max_courses: break
        if used_difficulty + numeric_diff > difficulty_budget: continue
        selected_rows.append(row)
        used_difficulty += numeric_diff
        
    return pd.DataFrame(selected_rows).reset_index(drop=True) if selected_rows else ordered.head(0).copy()

print("Todas as funções declaradas!")

### 3. Geração de Pares Sintéticos (Dataset de Treino)
O trabalho prático consiste em **treinar/fazer fine-tuning** de um modelo. Como não temos um dataset pronto de alunos, vamos gerar um dataset sintético baseado no fluxograma do PPC!
A função abaixo vai simular estudantes em vários períodos do curso e criar exemplos POSITIVOS (1.0) para matérias que ele pode puxar, e exemplos NEGATIVOS (0.0) para matérias bloqueadas ou muito fora do período.

In [ ]:
# Funções completas para gerar o Dataset de Fine-Tuning
def generate_sbert_training_pairs(disciplines_df):
    records = []
    discipline_codes = disciplines_df["codigo"].dropna().astype(str).tolist()
    discipline_lookup = disciplines_df.set_index("codigo")

    # Vamos simular perfis para alunos do período 1 até o 8
    max_period = int(disciplines_df["periodo_sugerido"].max())
    
    for period in range(1, 9):
        # Seleciona as disciplinas cursadas pelo aluno até aquele período
        completed_codes = disciplines_df.loc[disciplines_df["periodo_sugerido"] <= period, "codigo"].dropna().astype(str).tolist()
        if not completed_codes: continue
        
        # O texto A (Perfil do Aluno)
        text_a = get_completed_profile_text(disciplines_df, completed_codes)

        # O texto B (Disciplinas Candidatas)
        for code in discipline_codes:
            if code in completed_codes: continue
            
            row = discipline_lookup.loc[code]
            prereqs = set(row.get("pre_requisitos", []) or [])
            prereqs_met = prereqs.issubset(set(completed_codes))
            
            cand_period = row.get("periodo_sugerido")
            # É relevante se ele cumpriu os pré-requisitos E é do próximo período!
            relevant = bool(prereqs_met and pd.notna(cand_period) and int(cand_period) == period + 1)
            
            text_b = get_discipline_text(row)
            records.append({
                "text_a": text_a,
                "text_b": text_b,
                "label": 1.0 if relevant else 0.0,
            })

    return pd.DataFrame(records)

df_disciplinas = get_discipline_table()
print("Gerando pares sintéticos de treino a partir da grade da UFMG...")
df_treino = generate_sbert_training_pairs(df_disciplinas)
print(f"Dataset de treino gerado com {len(df_treino)} pares (perfil, disciplina)!")
display(df_treino.sample(5))

### 4. Fine-Tuning do Sentence-BERT (Treinamento)
Agora, vamos carregar o modelo pré-treinado do HuggingFace e **treiná-lo** usando nosso dataset gerado.
Utilizaremos a `CosineSimilarityLoss`, que ajusta os pesos da rede para aproximar o vetor do Perfil com o vetor da Disciplina se o `label = 1.0` (Relevante), e afastar se `label = 0.0` (Irrelevante).

In [ ]:
from sentence_transformers import InputExample, losses
from torch.utils.data import DataLoader

print("Carregando o modelo base SBERT...")
model = load_sbert_model()

# Convertendo para o formato do Sentence-Transformers
train_examples = []
for _, row in df_treino.iterrows():
    train_examples.append(InputExample(texts=[row['text_a'], row['text_b']], label=row['label']))

# Um batch_size de 4 consome cerca de 4GB-6GB de VRAM. No Google Colab funciona perfeitamente.
train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=4)
train_loss = losses.CosineSimilarityLoss(model)

print("Iniciando o Fine-Tuning do modelo...")
# Treinamos por 3 épocas. No Colab com GPU T4, isso leva poucos minutos.
model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=3, 
    warmup_steps=int(0.1 * len(train_dataloader)),
    show_progress_bar=True
)

print("TREINAMENTO CONCLUÍDO COM SUCESSO! O modelo agora está fine-tunado para a Engenharia de Sistemas.")

### 5. Execução do Modelo Treinado (Inferencia)
Agora que **treinamos** nosso próprio SBERT, vamos gerar os Embeddings de todas as disciplinas e testar simulando a dúvida de um aluno real!

In [ ]:
print("Gerando os novos embeddings com o modelo fine-tunado...")
corpus_embeddings = encode_disciplines(model, df_disciplinas)

# Simulação do Estudante
disciplinas_concluidas = ['DCC203', 'DCC204', 'MAT001', 'MAT039']
perfil_texto = get_completed_profile_text(df_disciplinas, disciplinas_concluidas)

# Pergunta em texto livre:
query_texto = perfil_texto + " Quero me aprofundar em análise de dados, inteligência artificial e estatística."
print(f"\nInput Natural do Aluno:\n{query_texto}\n")

query_embedding = encode_texts(model, [query_texto], batch_size=1)[0]
hits = semantic_search(query_embedding, corpus_embeddings, top_k=15)

candidatas_list = []
for hit in hits:
    idx = hit['index']
    row = df_disciplinas.iloc[idx].copy()
    row['predicted_probability'] = hit['score']
    row['difficulty'] = row['dificuldade_geral']
    candidatas_list.append(row)
df_candidatas = pd.DataFrame(candidatas_list)

# Regras de Pós-Filtro
df_validas = filter_recommended_candidates(df_candidatas, disciplinas_concluidas)
df_validas = df_validas[~df_validas['codigo'].isin(disciplinas_concluidas)]
df_recomendacao_final = greedy_balance_by_difficulty(df_validas, difficulty_budget=25.0, max_courses=4)

print("==== RECOMENDAÇÃO FINAL DO SEMESTRE (Baseada no modelo treinado) ====")
for _, row in df_recomendacao_final.iterrows():
    print(f"-> {row['codigo']} - {row['nome']} (Período: {row['periodo_sugerido']})")
    print(f"   Dificuldade: {row['difficulty']} | Score Semântico: {row['predicted_probability']:.4f}\n")
